# Глубокое обучение для компьютерного зрения


Эта тетрадка научит вас создавать и обучать сверточные нейронные сети для распознавания изображений. Приготовьтесь.

# Набор данных CIFAR
На этой неделе мы сосредоточимся на задаче распознавания изображений на наборе данных cifar10
* 60k изображений размером 3x32x32
* 10 различных классов: самолеты, собаки, кошки, грузовики и т.д.

<img src="https://github.com/yandexdataschool/deep_vision_and_graphics/blob/fall22/week02-convnets/cifar10.jpg?raw=1" style="width:80%">

In [ ]:
# **ВАЖНО** при запуске в colab, раскомментируйте эту строку
# !wget https://raw.githubusercontent.com/yandexdataschool/Practical_DL/refs/heads/fall25/week03_convnets/cifar.py

In [ ]:
import numpy as np
from cifar import load_cifar10
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10("cifar_data")

class_names = np.array(['самолет', 'автомобиль', 'птица', 'кошка', 'олень',
                        'собака', 'лягушка', 'лошадь', 'корабль', 'грузовик'])

print(X_train.shape,y_train.shape)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=[12,10])
for i in range(12):
    plt.subplot(3,4,i+1)
    plt.xlabel(class_names[y_train[i]])
    plt.imshow(np.transpose(X_train[i],[1,2,0]))

# Построение сети

Простые нейронные сети со слоями, применяемыми друг за другом, могут быть реализованы как `torch.nn.Sequential` — просто добавьте список готовых модулей и дайте ей обучиться.

Начнем с полносвязной сети в качестве базового решения (baseline), а затем будем постепенно ее улучшать.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

model = nn.Sequential(
    nn.Flatten(),  # меняем форму тензора с картинкой на вектор; если у вас есть сверточные / пулинг слои, они должны идти ДО flatten
    nn.Linear(3 * 32 * 32, 128),
    nn.ReLU(),
    nn.Linear(128, 10) # логиты для 10 классов. Softmax не используется - это сделано намеренно
)

Как и в нашем базовом туториале, мы будем обучать модель с помощью отрицательного логарифма правдоподобия, также известного как кросс-энтропия.

In [ ]:
def compute_loss(X_batch, y_batch):
    X_batch = torch.as_tensor(X_batch, dtype=torch.float32)
    y_batch = torch.as_tensor(y_batch, dtype=torch.int64)
    logits = model(X_batch)
    return F.cross_entropy(logits, y_batch).mean()

In [ ]:
# пример
compute_loss(X_train[:5], y_train[:5])

### Обучение на мини-батчах
* У нас есть 40 тысяч изображений, это слишком много для полнопакетного SGD. Давайте вместо этого обучаться на мини-батчах
* Ниже приведена функция, которая разбивает обучающую выборку на мини-батчи

In [ ]:
opt = torch.optim.SGD(model.parameters(), lr=0.01)

train_loss = []
val_accuracy = []

# Вспомогательная функция, которая возвращает мини-батчи для обучения нейронной сети
def iterate_minibatches(X, y, batchsize):
    indices = np.random.permutation(np.arange(len(X)))
    for start in range(0, len(indices), batchsize):
        ix = indices[start: start + batchsize]
        yield X[ix], y[ix]

In [ ]:
import time
num_epochs = 100 # общее количество полных проходов по обучающим данным
batch_size = 50  # количество примеров, обрабатываемых за одну итерацию SGD

for epoch in range(num_epochs):
    # В каждой эпохе мы делаем полный проход по обучающим данным:
    start_time = time.time()
    model.train(True) # включить режим обучения для dropout / batch_norm
    for X_batch, y_batch in iterate_minibatches(X_train, y_train, batch_size):
        # обучаемся на батче
        loss = compute_loss(X_batch, y_batch)
        loss.backward()
        opt.step()
        opt.zero_grad()
        train_loss.append(loss.item())  # .item() = преобразовать тензор с одним значением в float

    # И полный проход по валидационным данным:
    model.train(False)     # отключить dropout / использовать средние значения для batch_norm
    with torch.no_grad():  # не сохранять промежуточные активации
        for X_batch, y_batch in iterate_minibatches(X_val, y_val, batch_size):
            logits = model(torch.as_tensor(X_batch, dtype=torch.float32))
            y_pred = logits.argmax(-1).detach().numpy()
            val_accuracy.append(np.mean(y_batch == y_pred))

    # Затем мы выводим результаты для этой эпохи:
    print("Эпоха {} из {} заняла {:.3f}с".format(
        epoch + 1, num_epochs, time.time() - start_time))
    print("  потери на обучении (на итерации): \t{:.6f}".format(
        np.mean(train_loss[-len(X_train) // batch_size :])))
    print("  точность на валидации: \t\t\t{:.2f} %".format(
        np.mean(val_accuracy[-len(X_val) // batch_size :]) * 100))

Не ждите все 100 эпох. Вы можете прервать обучение через 5-20 эпох, как только точность на валидации перестанет расти.
```

```

```

```

```

```

```

```

```

```

### Финальный тест

In [ ]:
model.train(False) # отключить dropout / использовать средние значения для batch_norm
test_batch_acc = []
for X_batch, y_batch in iterate_minibatches(X_test, y_test, 500):
    logits = model(torch.as_tensor(X_batch, dtype=torch.float32))
    y_pred = logits.max(1)[1].data.numpy()
    test_batch_acc.append(np.mean(y_batch == y_pred))

test_accuracy = np.mean(test_batch_acc)

print("Финальные результаты:")
print("  точность на тесте:\t\t{:.2f} %".format(
    test_accuracy * 100))

if test_accuracy * 100 > 95:
    print("Перепроверьте, а затем подумайте о подаче заявки на NIPS'17. Серьезно.")
elif test_accuracy * 100 > 90:
    print("Вы чертовски круты!")
elif test_accuracy * 100 > 80:
    print("Достижение разблокировано: Чернокнижник 110-го уровня!")
elif test_accuracy * 100 > 70:
    print("Достижение разблокировано: Чернокнижник 80-го уровня!")
elif test_accuracy * 100 > 60:
    print("Достижение разблокировано: Чернокнижник 70-го уровня!")
elif test_accuracy * 100 > 50:
    print("Достижение разблокировано: Чернокнижник 60-го уровня!")
else:
    print("Нужно больше магии! Следуйте инструкциям ниже")

## Задание I: маленькая сверточная сеть
### Первый шаг

Давайте создадим мини-сверточную сеть примерно следующей архитектуры:
* Входной слой
* Свертка 3x3 с 10 фильтрами и активацией _ReLU_
* Пулинг 2x2 (или установите шаг (stride) предыдущей свертки равным 2)
* Слой Flatten (выпрямление)
* Полносвязный слой со 100 нейронами и активацией _ReLU_
* Dropout с вероятностью 10%
* Выходной полносвязный слой.


__Сверточные слои__ в torch такие же, как и все остальные слои, но с определенным набором параметров:

__`...`__

__`model.add_module('conv1', nn.Conv2d(in_channels=3, out_channels=10, kernel_size=3)) # свертка`__

__`model.add_module('pool1', nn.MaxPool2d(2)) # max pooling 2x2`__

__`...`__


Как только вы закончите (и `compute_loss` перестанет выдавать ошибки), обучите сеть с помощью оптимизатора __Adam__ с параметрами по умолчанию (можете изменять код выше).

Если все правильно, вы должны получить как минимум __50%__ точности на валидации.

```

```

```

```

```

```

```

```

```

__Подсказка:__ Если вы не хотите вычислять размеры вручную, просто подставьте любой размер (например, 1) и запустите `compute_loss`. Вы увидите что-то вроде этого:

__`RuntimeError: size mismatch, m1: [5 x 1960], m2: [1 x 64] at /some/long/path/to/torch/operation`__

Видите там __1960__? Это и есть реальный размер входа для полносвязного слоя.

## Задание 2: добавление нормализации

* Добавьте батч-нормализацию (с параметрами по умолчанию) между сверткой и ReLU
  * `nn.BatchNorm*d` (`1d` для полносвязных слоев, `2d` для сверточных)
  * обычно их лучше ставить после линейного/сверточного слоя, но перед нелинейностью
* Переобучите сеть с тем же оптимизатором, она должна достичь как минимум 60% точности на валидации на пике.




```

```

```

```

```

```

```

```

```

```

```

```

```
## Задание 3: Аугментация данных

Для предобработки изображений и аугментации данных в `torchvision` есть мощный инструмент `transforms`.

Вот как он работает: мы определяем конвейер (pipeline), который
* делает случайные вырезки (crop) из изображений (аугментация)
* случайно отражает изображение по горизонтали (аугментация)
* затем нормализует его (предобработка)

In [ ]:
from torchvision import transforms
means = np.array((0.4914, 0.4822, 0.4465))  # статистики из документации к набору данных
stds = np.array((0.2023, 0.1994, 0.2010))

transform_augment = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomRotation([-30, 30]),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(means, stds),
])

In [ ]:
from torchvision.datasets import CIFAR10
train_loader = CIFAR10("./cifar_data/", train=True, transform=transform_augment)

train_dataloader = torch.utils.data.DataLoader(
    train_loader,  batch_size=32, shuffle=True, num_workers=1)

In [ ]:

for (x_batch, y_batch) in train_dataloader:

    print('X:', type(x_batch), x_batch.shape)
    print('y:', type(y_batch), y_batch.shape)

    for i, img in enumerate(x_batch.numpy()[:8]):
        plt.subplot(2, 4, i+1)
        plt.imshow(img.transpose([1,2,0]) * stds + means )


    raise NotImplementedError("Пожалуйста, используйте этот код в вашем цикле обучения")
    # TODO: используйте это в вашем цикле обучения

При тестировании нам не нужны случайные вырезки, нужно просто нормализовать с теми же статистиками.

In [ ]:
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(means, stds),
])

test_loader = <ВАШ КОД>